# Custom function

In [1]:
#导入库
import numpy as np
import pandas as pd
import h5py
import collections
from sklearn.model_selection import KFold
import random
import warnings
from openpyxl import load_workbook
from openpyxl.styles import Font
import json
warnings.filterwarnings("ignore")
import torch
from torch import nn
import torch.nn.functional as F
from sklearn import preprocessing
from sklearn.metrics import pairwise_distances
import torch.optim as optim
from matplotlib import pyplot as plt
# 训练
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import matthews_corrcoef, average_precision_score
from sklearn.metrics import roc_curve, auc
from torch.optim import lr_scheduler

seed = 32
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed) 
np.random.seed(seed)
random.seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False


In [2]:
def calculate_metrics(y_true, y_pred):

    # Flatten the prediction and true values
    y_true = np.array(y_true)
    y_pred = np.array(y_pred).flatten()

    # Calculate ROC and AUC
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    auc_value = auc(fpr, tpr)
    aupr = average_precision_score(y_true, y_pred)

    # Find the optimal threshold using the ROC curve
    optimal_idx = np.argmax(tpr - fpr)
    optimal_threshold = thresholds[optimal_idx]
    y_pred_binary = (y_pred >= optimal_threshold).astype(int)

    # Calculate other metrics
    acc = accuracy_score(y_true, y_pred_binary)
    recall = recall_score(y_true, y_pred_binary)
    precision = precision_score(y_true, y_pred_binary)
    f1 = f1_score(y_true, y_pred_binary)

    # Return the calculated metrics
    return {
        "AUC": auc_value,
        "AUPR": aupr,
        "ACC": acc,
        "Recall": recall,
        "Precision": precision,
        "F1-Score": f1
    }

def save_results_to_excel(results, save_path, font_name='Times New Roman'):
  
        
    directory = os.path.dirname(save_path)
    if not os.path.exists(directory):
        os.makedirs(directory)

    results_df = pd.DataFrame(results)
    
  
    results_df.to_excel(save_path, index=False)

    wb = load_workbook(save_path)
    ws = wb.active

    
    average_rows = [
        idx + 2 for idx, row in results_df.iterrows() if row["Fold"] == "Average"
    ]

    for row in ws.iter_rows():
        for cell in row:
            cell.font = Font(name=font_name)  

    
    for cell in ws[1]:
        cell.font = Font(name=font_name, bold=True)

    for row_idx in average_rows:
        for cell in ws[row_idx]:
            cell.font = Font(name=font_name, bold=True)


    wb.save(save_path)

    print(f"Results have been saved to {save_path}.")

def load_h5_file(filepath):
    import h5py
    with h5py.File(filepath, 'r') as f:
        data = {key: f[key][()] for key in f.keys()}
    return data

In [3]:

def dag_loss(A, rho=1, alpha=1):
    d = A.size(0)
 
    A_hadamard = (A * A)/d

    exp_A = torch.matrix_exp(A_hadamard)

    trace = torch.trace(exp_A)

    hw = trace - d

    return (rho / 2) * hw*hw + alpha * hw


def reconstruction_loss(original_vector, reconstructed_vector):

    mse_loss = F.mse_loss(reconstructed_vector, original_vector)
    return mse_loss

def dynamic_loss(C_dynamic):
   
    T = C_dynamic.size(0)
    temporal_loss = 0.0
    for t in range(1, T):
        temporal_loss += torch.norm(C_dynamic[t] - C_dynamic[t-1], p='fro')
    temporal_loss /= (T - 1) 
    return temporal_loss

def consistency_loss(C_dynamic, C_static, F_dynamic, F_static):

  
    C_dynamic_mean = torch.mean(C_dynamic, dim=0)  # (n, n)

    
    causal_consistency_loss = torch.norm(C_dynamic_mean - C_static, p='fro')

   
    F_dynamic_health = F_dynamic[:, -1, :]  
    F_dynamic_disease = F_dynamic[:, -2, :]  

    F_static_health = F_static[-1, :] 
    F_static_disease = F_static[-2, :]  


    F_dynamic_health_mean = torch.mean(
        F_dynamic_health, dim=0)  
    F_dynamic_disease_mean = torch.mean(
        F_dynamic_disease, dim=0) 

    health_consistency_loss = torch.norm(
        F_dynamic_health_mean - F_static_health, p=2)
    disease_consistency_loss = torch.norm(
        F_dynamic_disease_mean - F_static_disease, p=2)
   
    total_loss = causal_consistency_loss + \
        disease_consistency_loss + health_consistency_loss
    return total_loss

def classify_loss(label, pred, alpha=0.25, gamma=2, reduction='mean'):


    loss = F.binary_cross_entropy(pred, label, reduction='none')

    p_t = label * pred + (1 - label) * (1 - pred)
    p_t = torch.clamp(p_t, 1e-7, 1.0)  
    modulating_factor = (1.0 - p_t) ** gamma
    loss *= modulating_factor

    if alpha > 0:
        alpha_factor = label * alpha + (1 - label) * (1 - alpha)
        loss *= alpha_factor

    if reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    else:
        return loss  
def Calculate_losses(label, prediction, causal_space1, causal_space2, causal_space3, weights):

    Ho1, Hd1, Hc1 = causal_space1
    Ho2, Hd2, Hc2 = causal_space2
    Ho3, Hd3, Hc3 = causal_space3
 
   
    w_rec = weights['w_rec']
    w_dag = weights['w_dag']
    w_dynamic = weights['w_dynamic']
    w_consistency = weights['w_consistency']

    L_cla = classify_loss(label, prediction)

    L_rec = reconstruction_loss(Ho1, Hd1)+reconstruction_loss(Ho2, Hd2)+torch.mean(
        torch.stack([reconstruction_loss(Ho3[t], Hd3[t]) for t in range(Ho3.size(0))])) 

    L_dag = dag_loss(Hc1)+dag_loss(Hc2)+torch.mean(torch.stack(
        [dag_loss(Hc3[t]) for t in range(Hc3.size(0))])) 

    L_dynamic = dynamic_loss(Hc3)

    L_consistency = consistency_loss(Hc3, Hc2, Hd3, Hd2)

    L_losses = L_cla + w_rec*L_rec+w_dag*L_dag+w_dynamic * \
        L_dynamic+w_consistency*L_consistency
    return L_losses

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader

class MyDataset2(Dataset):
    def __init__(self, data, SG=None, SC=None, label=None, lsa_data=None, 
                 flash_health=None, flash_disease=None, dynamic_health=None, dynamic_disease=None):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.SG = torch.tensor(SG, dtype=torch.float32) if SG is not None else None
        self.SC = torch.tensor(SC, dtype=torch.float32) if SC is not None else None
        self.label = torch.tensor(label, dtype=torch.float32) if label is not None else None
        self.lsa_data = torch.tensor(lsa_data, dtype=torch.float32) if lsa_data is not None else None
        
       
        self.flash_health = torch.tensor(flash_health, dtype=torch.float32) if flash_health is not None else None
        self.flash_disease = torch.tensor(flash_disease, dtype=torch.float32) if flash_disease is not None else None
        

        self.dynamic_health = torch.tensor(dynamic_health, dtype=torch.float32) if dynamic_health is not None else None
        self.dynamic_disease = torch.tensor(dynamic_disease, dtype=torch.float32) if dynamic_disease is not None else None

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
       
        return (
            self.data[index], 
            self.SG[index] if self.SG is not None else None, 
            self.SC[index] if self.SC is not None else None, 
            self.label[index] if self.label is not None else None,
            self.lsa_data[index] if self.lsa_data is not None else None, 
            self.flash_health,  
            self.flash_disease, 
            self.dynamic_health, 
            self.dynamic_disease 
        )
def collate_fn2(batch):

    data, SG, SC, label, lsa_data, flash_health, flash_disease, dynamic_health, dynamic_disease = zip(*batch)
    
   
    data = torch.stack(data)
    label = torch.stack(label) if label[0] is not None else None
    lsa_data = torch.stack(lsa_data) if lsa_data[0] is not None else None

   
    SG = torch.stack(SG) if SG[0] is not None else None
    SC = torch.stack(SC) if SC[0] is not None else None
    
   
    flash_health = flash_health[0] 
    flash_disease = flash_disease[0]  
    

    dynamic_health = dynamic_health[0] 
    dynamic_disease = dynamic_disease[0]  

    return (data, SG, SC, label, lsa_data, flash_health, flash_disease, dynamic_health, dynamic_disease)


In [5]:
import torch
from torch_geometric.data import Data

def create_graph_data(weight_matrix):
  
   
    if weight_matrix.dim() != 3:
        raise ValueError("(num_graphs, num_nodes, num_nodes)")

    num_graphs = weight_matrix.shape[0]
    data_list = []

    for i in range(num_graphs):
     
        edge_index = torch.nonzero(weight_matrix[i] != 0).T  
        
      
        edge_attr = weight_matrix[i][edge_index[0], edge_index[1]]  
        

        x = weight_matrix[i] 


        data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
        data_list.append(data)

    return data_list


# Training function

In [7]:

def classify_train(model, Epoch_num, Batch_size, learning_rate, train_loader, test_loader, weights):

    train_losses = []
    epochs = []
    train_aucs = []
    train_accs = []
    train_auprs = []
    train_recalls = []
    train_precisions = []
    train_f1s = []    
    val_losses = []
    val_aucs = []
    val_auprs = []
    val_accs = []
    val_precisions = []
    val_recalls = []
    val_f1s = []

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate,weight_decay=1e-5)#
    scheduler = lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.1)

    model.train()
    best_auc = float('-inf') 
    best_model = None
    for epoch in range(Epoch_num):
        model.train()
        optimizer.zero_grad()  # 梯度初始化为零
        train_loss = 0.0     
        train_auc = 0.0
        train_acc = 0.0
        train_aupr = 0.0
        train_recall = 0.0
        train_precision = 0.0
        train_f1 = 0.0
        for step, (batch_x, batch_SG, batch_SC, batch_label,batch_lsa,batch_ctrl,batch_case,batch_dyctrl,batch_dycase) in enumerate(train_loader):
            train_input = batch_x.to(device)
            y_true = batch_label.to(device)
          
            train_SG = batch_SG.to(device) if batch_SG is not None else None
            train_SC = batch_SC.to(device) if batch_SC is not None else None
            batch_lsa = batch_lsa.to(device)  #
            batch_ctrl = batch_ctrl.to(device)  # 
            batch_case = batch_case.to(device)  # 
            batch_ctrl_dynamic = batch_dyctrl.to(device)  # 
            batch_case_dynamic = batch_dycase.to(device)  # 

            space1, space2, space3, y_pred = model(train_input, train_SG, train_SC, y_true, batch_lsa, batch_ctrl, batch_case,batch_ctrl_dynamic,batch_case_dynamic)
            
            loss =  Calculate_losses(y_true, y_pred, space1, space2, space3, weights)
 
            
            loss.backward()
    
            for module in model.modules():
                if isinstance(module, CausalBlock):
                    module.zero_diagonal_grad()
            optimizer.step()  
            train_loss += loss.item()
            
           
            metrics = calculate_metrics(y_true.detach().cpu().numpy(), y_pred.detach().cpu().numpy())
            

            tra_auc = metrics["AUC"]
            tra_aupr = metrics["AUPR"]
            tra_acc = metrics["ACC"]
            tra_precision = metrics["Recall"]#, average='binary'
            tra_recall = metrics["Precision"]
            tra_f1 = metrics["F1-Score"]
            
            train_auc += tra_auc
            train_aupr += tra_aupr
            train_acc += tra_acc
            train_precision += tra_precision
            train_recall += tra_recall
            train_f1 += tra_f1

        train_loss /= len(train_loader) 
        scheduler.step()
        
        train_auc /= len(train_loader)
        train_aupr /= len(train_loader)
        train_acc /= len(train_loader)
        train_recall /= len(train_loader)
        train_precision /= len(train_loader)
        train_f1 /= len(train_loader)

        val_loss, val_auc, val_acc, val_aupr, val_recall, val_precision, val_f1 = classify_test(
            model, Batch_size, test_loader,weights)

        val_losses.append(val_loss)
        val_aucs.append(val_auc)
        val_accs.append(val_acc)
        val_auprs.append(val_aupr)
        val_recalls.append(val_recall)
        val_precisions.append(val_precision)
        val_f1s.append(val_f1)
        
        if val_auc > best_auc:
            best_auc = val_auc
            best_model = model.state_dict()


        model.load_state_dict(best_model)
        
 
        train_losses.append(train_loss)
        epochs.append(epoch)
        train_aucs.append(train_auc)
        train_auprs.append(train_aupr)
        train_accs.append(train_acc)
        train_recalls.append(train_recall)
        train_precisions.append(train_precision)
        train_f1s.append(train_f1)
        if epoch % 50 == 0: 
            print(f'Epoch: {epoch}, '
                  f'train_Loss: {train_loss:.5f}, '
                  f'train_AUC: {train_auc:.5f}, '
                  f'train_AUPR: {train_aupr:.5f}, '
                  f'train_ACC: {train_acc:.5f}, '
                  f'train_Precision: {train_precision:.5f},'
                  f'train_ Recall: {train_recall:.5f}, '
                  f'train_F1: {train_f1:.5f}\n ',
                  f'val_Loss: {val_loss:.5f}, '
                  f'val_AUC: {val_auc:.5f}, '
                  f'val_AUPR: {val_aupr:.5f}, '
                  f'val_ACC: {val_acc:.5f}, '
                  f'val_Precision: {val_precision:.5f}, '
                  f'val_Recall: {val_recall:.5f}, '
                  f'val_F1: {val_f1:.5f}')    
   
    model.load_state_dict(best_model)

    if test_loader is not None:
        test_loss, test_auc,test_aupr, test_acc,test_recall,test_precision,test_f1 = classify_test(model, Batch_size, test_loader,weights)
    else:
        test_loss, test_auc,test_aupr, test_acc,test_recall,test_precision,test_f1 = None, None, None, None, None, None, None

    return test_loss, test_auc,test_aupr,test_acc,test_recall,test_precision,test_f1


def classify_test(model, Batch_size, test_loader,weights):  

    model = model.to(device)
    model.eval()
    with torch.no_grad():  
  
        test_auc = 0.0
        test_acc = 0.0
        test_aupr = 0.0
        test_recall = 0.0
        test_precision = 0.0
        test_f1 = 0.0
        for step, (batch_x, batch_SG, batch_SC, batch_label,batch_lsa,batch_ctrl,batch_case,batch_dyctrl,batch_dycase) in enumerate(test_loader):
            test_input = batch_x.to(device)
            y_true = batch_label.to(device)
          
            test_SG = batch_SG.to(device) if batch_SG is not None else None
            test_SC = batch_SC.to(device) if batch_SC is not None else None
            test_lsa = batch_lsa.to(device)      
            test_ctrl = batch_ctrl.to(device)     
            test_case = batch_case.to(device)     
            batch_ctrl_dynamic = batch_dyctrl.to(device)  # 
            batch_case_dynamic = batch_dycase.to(device)  # 

            space1, space2, space3, y_pred = model(test_input, test_SG, test_SC, y_true, test_lsa, test_ctrl, test_case, batch_ctrl_dynamic,batch_case_dynamic)
            
            loss =  Calculate_losses(y_true, y_pred, space1, space2, space3, weights)
       
            test_loss = loss.item()     
    
      
            metrics = calculate_metrics(y_true.detach().cpu().numpy(), y_pred.detach().cpu().numpy())
            val_auc = metrics["AUC"]
            val_aupr = metrics["AUPR"]
            val_acc = metrics["ACC"]
            val_precision = metrics["Recall"]
            val_recall = metrics["Precision"]
            val_f1 = metrics["F1-Score"]    
    
            test_auc += val_auc
            test_aupr += val_aupr
            test_acc += val_acc
            test_precision += val_precision
            test_recall += val_recall
            test_f1 += val_f1
        
        test_loss /= len(test_loader)        
        test_auc /= len(test_loader)
        test_aupr /= len(test_loader)
        test_acc /= len(test_loader)
        test_recall /= len(test_loader)
        test_precision /= len(test_loader)
        test_f1 /= len(test_loader)
                  
    return test_loss, test_auc, test_aupr, test_acc, test_recall, test_precision, test_f1

# Model

In [8]:
import torch_geometric.nn as gnn
from torch_geometric.data import Batch
from scipy import stats


def clr_transform(X):
    X = X+1
    
    geometric_mean = X.prod(dim=1)**(1.0/X.size(1))

    geometric_mean = geometric_mean.unsqueeze(1)
  
    ratio = X / geometric_mean

    log_ratio = torch.log(ratio)
    return log_ratio


class BiLSTM(nn.Module):
    def __init__(self, input_size, output_size, num_layers):
        super(BiLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, output_size//2,
                            num_layers, batch_first=True, bidirectional=True)
        self.layer_norm = nn.LayerNorm(output_size)
        self.fc = nn.Linear(output_size, output_size)

    def attention_net(self, lstm_output, final_state):
        batch_size = len(lstm_output)
        # hidden : [batch_size, n_hidden * num_directions(=2), n_layer(=1)]
        hidden = final_state.view(batch_size, -1, 1)
        attn_weights = torch.bmm(lstm_output, hidden).squeeze(
            2)  # attn_weights : [batch_size, n_step]
        soft_attn_weights = F.softmax(attn_weights, 1)
        # context : [batch_size, n_hidden * num_directions(=2)]
        context = torch.bmm(lstm_output.transpose(
            1, 2), soft_attn_weights.unsqueeze(2)).squeeze(2)
        return context, soft_attn_weights

    def forward(self, x):
        out, (h, c) = self.lstm(x) 
        out = self.layer_norm(out)
        attn, _ = self.attention_net(out, h)
        out = self.fc(attn)  
        return out


class GCNModel(nn.Module):
    def __init__(self, input_dim, output_dim, dropout_rate):
        super(GCNModel, self).__init__()
        self.conv1 = gnn.GCNConv(input_dim, output_dim//4)
        self.conv2 = gnn.GCNConv(output_dim//4, output_dim//2)
        self.dropout = nn.Dropout(dropout_rate)  


        self.fc = nn.Linear(output_dim//2, output_dim)  

    def forward(self, data_list):

        batch = Batch.from_data_list(data_list)

        x, edge_index = batch.x, batch.edge_index

        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        x = torch.relu(x)
        x = self.dropout(x)


        x = gnn.global_mean_pool(x, batch.batch)


        x = self.fc(x)
        x = torch.relu(x)

        return x  



class LinearTransform(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(LinearTransform, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.linear(x)


class SharedEncoder(nn.Module):
    def __init__(self, input_size, dropout_rate):
        super(SharedEncoder, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_size, input_size//2),
            #             nn.LayerNorm(input_size//2),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),   
            nn.Linear(input_size//2, input_size//4),
            #             nn.LayerNorm(input_size//4),
            nn.ReLU(),
            # nn.Dropout(p=dropout_rate)   

    def forward(self, x):
        return self.mlp(x)


class SharedDecoder(nn.Module):
    def __init__(self, input_size, dropout_rate):
        super(SharedDecoder, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_size//4, input_size//4),
            #             nn.LayerNorm(input_size//4),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),  
            nn.Linear(input_size//4, input_size//2),
            #             nn.LayerNorm(input_size//2),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),  
            nn.Linear(input_size//2, input_size),
            #             nn.LayerNorm(input_size),
            nn.ReLU(),
            # nn.Dropout(p=dropout_rate),
        )

    def forward(self, x):
        return self.mlp(x)


class LabelDecoder(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout_rate):
        super(LabelDecoder, self).__init__()
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(input_size+hidden_size, output_size)
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, x):
        out = F.relu(self.linear1(x)) 
        out = self.dropout(out) 
        out = torch.cat((x, out), dim=1)  
        out = F.sigmoid(self.linear2(out))  
        return out

class TimeCausalMatrix(nn.Module):
    def __init__(self, size):
        super(TimeCausalMatrix, self).__init__()
        self.timecausal_mat = nn.Parameter(torch.randn(size, size))
        
class CausalBlock(nn.Module):
    def __init__(self, causal_size, linear_size, block_type,iftime):
        super(CausalBlock, self).__init__()
        self.causal_size = causal_size
        self.linear_size = linear_size
        self.block_type = block_type
        self.iftime = iftime
        self.causal_mat = nn.Parameter(torch.randn(
            causal_size, causal_size))  
                    
        if self.iftime == True:
            self.causal_time = nn.ModuleList([TimeCausalMatrix(causal_size) for _ in range(10)])
        
        if self.block_type == 'semantic':
            self.mask = torch.ones(causal_size, causal_size) - \
                torch.eye(causal_size)  
        elif self.block_type == 'feature':
            self.mask = torch.ones(causal_size, causal_size) - \
                torch.eye(causal_size) 
            self.mask[-1, -2] = 0  
            self.mask[-2, -1] = 0  
        self.linear_transforms = nn.ModuleList(
            [nn.Linear(linear_size, linear_size) for _ in range(causal_size)])

    def forward(self, x, causal_size=None,time_steps=None):
              
        causal_size = causal_size if causal_size is not None else self.causal_size
#
        h_prime = []
        for i in range(causal_size):
            transformed = self.linear_transforms[i](x[i])
            h_prime.append(transformed)
       
        h_prime = torch.stack(h_prime, dim=0)  # 【6,32,128】

        causal_mat = self.causal_mat[:causal_size, :causal_size].to(x.device)  
       
        if time_steps is not None and self.iftime == True:
#             print("Time",time_steps)
            causal_mat = causal_mat + self.causal_time[time_steps].timecausal_mat[:causal_size, :causal_size].to(x.device)
        mask = self.mask[:causal_size, :causal_size].to(x.device) 
        
        causal_mat = causal_mat * mask  # 

        h_prime_new = torch.einsum('ik, i... -> k...', causal_mat, h_prime)

        return h_prime_new, causal_mat

    def zero_diagonal_grad(self):
     
        if self.causal_mat.grad is not None:
            with torch.no_grad():

                grad = self.causal_mat.grad
                diag_indices = torch.arange(
                    self.causal_size, device=grad.device)

                if self.block_type == 'semantic':
   
                    grad[diag_indices, diag_indices] = 0

                elif self.block_type == 'feature':
                   
                    grad[-1, -2] = 0  # 
                    grad[-2, -1] = 0  # 

                    grad[diag_indices, diag_indices] = 0 
        else:
      
            print("Warning: causal_mat.grad is None. No gradient to modify.")

class SemSCM(nn.Module):
    def __init__(self, lstm_params, gcn_params, linear_params, clssify_params, dropout):
        super(SemSCM, self).__init__()
        self.dropout = dropout
        self.output_size = lstm_params['output_size'] 
        # input_size, hidden_size, output_size，num_layers
        self.bilstm = BiLSTM(**lstm_params)
        self.gcn = GCNModel(**gcn_params, dropout_rate=self.dropout)
        self.linear1 = LinearTransform(
            linear_params['N_gender'], self.output_size)  # sg
        self.linear2 = LinearTransform(
            linear_params['N_country'], self.output_size)  # sc
        self.lineary = LinearTransform(
            linear_params['N_label'], self.output_size)  # sy
        self.encoder = SharedEncoder(
            input_size=self.output_size, dropout_rate=self.dropout)  
        self.decoder = SharedDecoder(
            input_size=self.output_size, dropout_rate=self.dropout)  

        self.causal_size = 5  
        self.causal = CausalBlock(
            causal_size=self.causal_size, linear_size=self.output_size//4, block_type='semantic',iftime=False)

    def forward(self, *inputs):
        input_sto, input_sg, input_sc, input_sy, input_lsa, _, _, _, _ = inputs
       
        batch_size, time, otu = input_sto.size()

        input_clr = clr_transform(input_sto.reshape(-1, otu))  
        input_clr = input_clr.view(batch_size, time, otu)


        graph_lsa = create_graph_data(input_lsa)

        if input_sg is None or input_sc is None:
            current_causal_size = 4  
        else:
            current_causal_size = self.causal_size  
        

        Hs = self.bilstm(input_clr)  
        Ho = self.gcn(graph_lsa)
        Hg = self.linear1(input_sg) if input_sg is not None else None
        Hc = self.linear2(input_sc) if input_sc is not None else None
        Hy = self.lineary(input_sy)

        H_list = [Hs, Ho, Hc, Hg, Hy]  

       
        H_list = [h for h in H_list if h is not None]

  
        H_original = torch.stack(H_list, dim=0)
        

        H_encoded = self.encoder(H_original)  
        H_causal_rec, H_causal_mat = self.causal(H_encoded, causal_size=current_causal_size)
        H_decoded = self.decoder(H_causal_rec)
        Hy_decoded = H_decoded[-1] 

        return H_original, H_decoded, H_causal_mat, Hy_decoded  # ,label_decoded


class SharedComponents(nn.Module):
    def __init__(self, lstm_params, gcn_params, dropout, iftime=False):
        super(SharedComponents, self).__init__()
        self.output_size = lstm_params['output_size']
        self.gcn = GCNModel(**gcn_params, dropout_rate=dropout)

        self.encoder = SharedEncoder(input_size=self.output_size, dropout_rate=dropout)
        self.decoder = SharedDecoder(input_size=self.output_size, dropout_rate=dropout)
        self.causal = CausalBlock(
            causal_size=N_OTU + 2, linear_size=self.output_size // 4, block_type='feature', iftime=iftime)

    def forward(self, input_data, graph_disease, graph_health, time_steps=None):
        # GCN processing
        Hf = input_data
        Hh = self.gcn(graph_health)
        Hd = self.gcn(graph_disease)
        
        # Concatenate
        H_original = torch.cat([Hf, Hd, Hh], dim=0)

        # Encoding and causal inference
        H_encoded = self.encoder(H_original)
        H_causal_rec, H_causal_mat = self.causal(H_encoded)
        H_decoded = self.decoder(H_causal_rec)

        return H_original, H_decoded, H_causal_mat# 二维或者三维


class StaticFeaSCM(nn.Module):
    def __init__(self, lstm_params, gcn_params, linear_params, dropout):
        super(StaticFeaSCM, self).__init__()
        self.dropout = dropout
        self.output_size = lstm_params['output_size']
        self.shared = SharedComponents(lstm_params, gcn_params, dropout, iftime=False)
  
        self.max_size = 3000  
        self.linear1 = nn.Linear(in_features=self.max_size, out_features=self.output_size)
    def forward(self, *inputs):
        input_sto, _, _, _, _, input_ctrl, input_case, _, _ = inputs
        batch_size, time, feature_d = input_sto.size()

      
        merged_x = input_sto.view(-1, feature_d)
        merged_x = merged_x.transpose(0, 1) 

        actual_input_size = min(merged_x.size(1), self.max_size)
        
       
        Hf  = torch.nn.functional.linear(merged_x, self.linear1.weight[:, :actual_input_size], self.linear1.bias)


        # Create graph data for control and case groups
        graph_flash_health = create_graph_data(input_ctrl)
        graph_flash_disease = create_graph_data(input_case)
        
#         print("Shape of graph_flash_health:", len(graph_flash_health),graph_flash_health[0])
#         print("Shape of graph_flash_disease:", len(graph_flash_disease),graph_flash_disease[0])
        
        # Use shared components
        H_original, H_decoded, H_causal_mat = self.shared(Hf, graph_flash_disease, graph_flash_health)

        return H_original, H_decoded, H_causal_mat


class DynamicFeaSCM(nn.Module):
    def __init__(self, lstm_params, gcn_params, linear_params, dropout):
        super(DynamicFeaSCM, self).__init__()
        self.dropout = dropout
        self.output_size = lstm_params['output_size']
        self.shared = SharedComponents(lstm_params, gcn_params, dropout,iftime=True)
        self.max_size = 300  
        self.shared_linear = nn.Linear(in_features=self.max_size, out_features=self.output_size)
        
    def forward(self, *inputs):
        input_sto, _, _, _, _, _, _, input_dyn_ctrl, input_dyn_case = inputs
        batch_size, time, feature_d = input_sto.size()
     

        input_x = input_sto.permute(1, 0, 2).permute(0, 2, 1)

        Hft = torch.nn.functional.linear(input_x, self.shared_linear.weight[:, :batch_size], self.shared_linear.bias)
        # Create graph data for dynamic control and case groups
        graph_flash_health_dyn = create_graph_data(input_dyn_ctrl)
        graph_flash_disease_dyn = create_graph_data(input_dyn_case)
        dynamic_health = self.shared.gcn(graph_flash_health_dyn).unsqueeze(1) 
        dynamic_disease = self.shared.gcn(graph_flash_disease_dyn).unsqueeze(1) 

        # Use shared components
        H_original = torch.cat([Hft, dynamic_disease, dynamic_health], dim=1)

        # Unbind along the time dimension for each time step
        H_original_list = torch.unbind(H_original, dim=0)

        H_decoded_list = []
        H_causal_mat_list = []

        # Process each time step
        for t in range(len(H_original_list)):
            H_encoded = self.shared.encoder(H_original_list[t])
            H_causal_rec, H_causal_mat = self.shared.causal(H_encoded, time_steps=t)#使用时间特异性因果
            H_decoded = self.shared.decoder(H_causal_rec)

            H_decoded_list.append(H_decoded)
            H_causal_mat_list.append(H_causal_mat)

       
        H_decoded_final = torch.stack(H_decoded_list, dim=0)
        H_causal_mat_final = torch.stack(H_causal_mat_list, dim=0)

        return H_original, H_decoded_final, H_causal_mat_final
    
class DCSM(nn.Module):
    def __init__(self, lstm_params, gcn_params, linear_params, classify_params, dropout):
        super(DCSM, self).__init__()
        self.dropout = dropout
        self.semcm = SemSCM(lstm_params, gcn_params,
                            linear_params, classify_params, self.dropout)
        self.static = StaticFeaSCM(
            lstm_params, gcn_params, linear_params, self.dropout)
        self.dynamic = DynamicFeaSCM(
            lstm_params, gcn_params, linear_params, self.dropout)
        self.label_decoder = LabelDecoder(
            **classify_params, dropout_rate=self.dropout)  

    def forward(self, *inputs):
        H_o1, H_d1, H_c1, Hy = self.semcm(*inputs) 
        H_o2, H_d2, H_c2 = self.static(*inputs)
        H_o3, H_d3, H_c3 = self.dynamic(*inputs)
        prediction = self.label_decoder(Hy)
        return (H_o1, H_d1, H_c1), (H_o2, H_d2, H_c2), (H_o3, H_d3, H_c3),prediction

# 加载数据

In [17]:
import os


base_dir = "Code and data/Data/SysLM-C-input"#

dataset_names = ['BONUS-CF']
taxonomies = ['P']  # P,C,O,F,G,S

dataset_classification_mapping = {

    'BONUS-CF': ['Cysticfibrosis'],
}

data_store = {}

print("Loading all datasets into memory...")


for dataset_name in dataset_names:

    if dataset_name not in dataset_classification_mapping:
        print(f"Dataset {dataset_name} has no classification labels defined. Skipping...")
        continue  


    classification_labels = dataset_classification_mapping[dataset_name]


    for classification_label in classification_labels:

        for taxonomy in taxonomies:

            for fold_number in range(1, 6):
                
                hdf5_lsa = os.path.join(base_dir, '/LSA/SysLM-I',dataset_name, classification_label, taxonomy, f'LSA_weights_fold_{fold_number}.h5')
                hdf5_flashwave_static = os.path.join(base_dir, '/FlashWave/interaction/SysLM-I', dataset_name, classification_label, taxonomy, f'static_weights_fold_{fold_number}.h5')
                hdf5_flashwave_dynamic = os.path.join(base_dir, '/FlashWave/interaction/SysLM-I', dataset_name, classification_label, taxonomy, f'dynamic_weights_fold_{fold_number}.h5')
                hdf5_train_test = os.path.join(base_dir, '/SysLM-I/', dataset_name, classification_label, taxonomy, f'train_test_data_fold_{fold_number}.h5')

              
                if not os.path.exists(hdf5_lsa) or not os.path.exists(hdf5_flashwave_static) or not os.path.exists(hdf5_flashwave_dynamic) or not os.path.exists(hdf5_train_test):
                    print(f" files for fold_{fold_number} {taxonomy} under {dataset_name}, {classification_label} are missing. Skipping taxonomy {taxonomy}...")
                    continue  

                fold_data = {}

                
                try:
                    
                    fold_data['lsa'] = load_h5_file(hdf5_lsa)
                    fold_data['flashwave_static'] = load_h5_file(hdf5_flashwave_static)
                    fold_data['flashwave_dynamic'] = load_h5_file(hdf5_flashwave_dynamic)
                    fold_data['train_test'] = load_h5_file(hdf5_train_test)  

                    data_store[(dataset_name, classification_label, taxonomy, fold_number)] = fold_data

#                     print(f"Loaded fold {fold_number} data for {dataset_name}, {classification_label}, {taxonomy}")
                
                except FileNotFoundError as e:
                    print(f"Error loading file: {e}. Skipping this fold...")
                    continue  
print("All datasets loaded into memory.")


Loading all datasets into memory...
All datasets loaded into memory.


# cross validation

In [18]:
# 定义权重字典
loss_weights = {
    'Combo_1': {                 # 0.5-1.5，total，
        'w_rec': 1e-4,          
        'w_dag': 1e-1,        
        'w_dynamic': 1e-2,    #
        'w_consistency': 1e-3 # 
    },
    'Combo_2': {                  # 1.5-3
        'w_rec': 1e-3,         # 
        'w_dag': 1e-5,         # 
        'w_dynamic': 1e-1,     # 
        'w_consistency': 1e-2  # 
    },
    'Combo_3': {                   # 3-10，
        'w_rec': 1e-1,          # 
        'w_dag': 1e-5,         # 
        'w_dynamic': 1e-2,     # 
        'w_consistency': 1e-1  # 
    },
    'Combo_4': {                 # <0.5or>10，
        'w_rec': 1e-1,         #
        'w_dag': 1e-4,         # 
        'w_dynamic': 1e-2,     # 
        'w_consistency': 1e-5  # 
    }
}

In [19]:
loss_weights_map = {
    'total': loss_weights['Combo_1'],
    'milk': loss_weights['Combo_2'],#97/40=2.43
    'egg': loss_weights['Combo_3'],#108/29=3.72
    'peanut': loss_weights['Combo_4'],#129/8=16.13    
    'Cysticfibrosis': loss_weights['Combo_4'],#25/193=0.13
    'preterm': loss_weights['Combo_2'],#24/8=3
    'changed': loss_weights['Combo_4'],#24/155=0.154
    'pregnant': loss_weights['Combo_2'],#30/16=1.88
    'cd': loss_weights['Combo_1'],#44/37=1.19
    'uc': loss_weights['Combo_2'],#59/22=2.69
    'nonibd': loss_weights['Combo_2'],#59/22=2.69
    'label': loss_weights['Combo_1'],  #10/13=0.77
}

In [20]:
import gc
import time
# Create combinations of all weights
from itertools import product

Epoch_num = 200
Batch_size = 256
learning_rate = 0.001  # [0.01,0.001,0.0001,0.00001]

N_representation = 64


global_temp_save_path =f"/Classify_temp_results_all.json"

global_dir_path = os.path.dirname(global_temp_save_path)
os.makedirs(global_dir_path, exist_ok=True)

    # Load global temporary results if available
if os.path.exists(global_temp_save_path):
    with open(global_temp_save_path, 'r') as f:
        all_results = json.load(f)
else:
    all_results = []

    # Record completed tasks to avoid reprocessing
completed_tasks = {(res["Dataset"], res["Classification_Label"], res["Taxonomy"], res["Fold"]) for res in all_results}

# Loop over each dataset
for dataset_name in dataset_names:
    if dataset_name not in dataset_classification_mapping:
        continue  # Skip datasets without classification labels

    classification_labels = dataset_classification_mapping[dataset_name]
                
    temp_save_path = f"/{dataset_name}/Classify_temp_results.json"
          
    dir_path = os.path.dirname(temp_save_path)
    os.makedirs(dir_path, exist_ok=True)

  
    if os.path.exists(temp_save_path):
        with open(temp_save_path, 'r') as f:
            dataset_results = json.load(f)
    else:
        dataset_results = []

            # Record completed tasks for this dataset
    completed_tasks_for_dataset = {(res["Classification_Label"], res["Taxonomy"], res["Fold"]) for res in dataset_results}
    
    for classification_label in classification_labels: 

        for taxonomy in taxonomies:
              
            key_to_check = (dataset_name, classification_label, taxonomy)
            matched = any(key[:3] == key_to_check for key in data_store.keys())

            if not matched:
                print(f"Taxonomy {taxonomy} does not exist for {dataset_name}, {classification_label}. Skipping...")
                continue
                    
            if classification_label in loss_weights_map:
                loss_weights_sets = loss_weights_map[classification_label]
            else:
                print(f"Warning: No loss weights set found for {classification_label}. Skipping...")
                continue
 

                
                                       
            print(f"Running classification for {dataset_name},{classification_label}, taxonomy {taxonomy} and {loss_weights_sets}...")
                    
            torch.cuda.empty_cache()
            gc.collect()  
                    
            results = []
            fold_times = []  # To store the runtime of each fold
                        # Loop over each combination of loss_weights
                    # Loop over each fold (1 to 5)
                        # Loop over each fold (1 to 5)
            for fold_number in range(1, 6):
                        # Check if the task has already been completed
                if (classification_label, taxonomy, fold_number) in completed_tasks_for_dataset:
#                         print(f"Skipping completed task: {classification_label}, {taxonomy}, Fold {fold_number}")
                    continue

                        # Experiment logic goes here
                print(f"Dataset {dataset_name}, Running Fold {fold_number} for {classification_label} and taxonomy {taxonomy}...")

                fold_data = data_store[(
                            dataset_name, classification_label, taxonomy, fold_number)]
                train_test_data = fold_data['train_test']
                lsa_data = fold_data['lsa']
                flashwave_static_data = fold_data['flashwave_static']
                flashwave_dynamic_data = fold_data['flashwave_dynamic']

                train_SC = train_test_data['train_SC'][:] if 'train_SC' in train_test_data else None
                train_SG = train_test_data['train_SG'][:] if 'train_SG' in train_test_data else None
                train_data = train_test_data['train_data'][:] if 'train_data' in train_test_data else None
                train_label = train_test_data['train_label'][:] if 'train_label' in train_test_data else None

                test_SC = train_test_data['test_SC'][:] if 'test_SC' in train_test_data else None
                test_SG = train_test_data['test_SG'][:] if 'test_SG' in train_test_data else None
                test_data = train_test_data['test_data'][:] if 'test_data' in train_test_data else None
                test_label = train_test_data['test_label'][:] if 'test_label' in train_test_data else None

                train_lsa = lsa_data['train_lsa']
                train_lsa = np.transpose(train_lsa, (2, 0, 1))

                test_lsa = lsa_data['test_lsa']
                test_lsa = np.transpose(test_lsa, (2, 0, 1))

                train_static_health = flashwave_static_data['train_healthy']
                train_static_disease = flashwave_static_data['train_disease']
                test_static_health = flashwave_static_data['test_healthy']
                test_static_disease = flashwave_static_data['test_disease']

                        # Using np.expand_dims to add a new dimension
                train_static_health = np.expand_dims(train_static_health, axis=0)
                train_static_disease = np.expand_dims(train_static_disease, axis=0)
                test_static_health = np.expand_dims(test_static_health, axis=0)
                test_static_disease = np.expand_dims(test_static_disease, axis=0)

                train_dynamic_health = flashwave_dynamic_data['train_healthy']
                train_dynamic_disease = flashwave_dynamic_data['train_disease']
                test_dynamic_health = flashwave_dynamic_data['test_healthy']
                test_dynamic_disease = flashwave_dynamic_data['test_disease']

                        # Create the DataLoader instances
                train_torch_dataset = MyDataset2(train_data, train_SG, train_SC, train_label, train_lsa,
                                                         train_static_health, train_static_disease, train_dynamic_health, train_dynamic_disease)
                train_loader = DataLoader(dataset=train_torch_dataset, batch_size=Batch_size,
                                                  shuffle=True, num_workers=0, collate_fn=collate_fn2)

                test_torch_dataset = MyDataset2(test_data, test_SG, test_SC, test_label, test_lsa,
                                                        test_static_health, test_static_disease, test_dynamic_health, test_dynamic_disease)
                test_loader = DataLoader(dataset=test_torch_dataset, batch_size=Batch_size,
                                                 shuffle=True, num_workers=0, collate_fn=collate_fn2)

                        # Network parameters
                N_OTU = train_data.shape[2]
                N_gender = train_SG.shape[1]if train_SG is not None else 1
                N_country = train_SC.shape[1]if train_SC is not None else 1
                N_label= train_label.shape[1]

                lstm_params = {'input_size': N_OTU, 'output_size': N_representation, 'num_layers': 1}
                gcn_params = {'input_dim': N_OTU, 'output_dim': N_representation}
                linear_params = {'N_label': N_label, 'N_country': N_country, 'N_gender': N_gender}
                classify_params = {'input_size': N_representation, "hidden_size": N_representation // 2, 'output_size': 1}

                        # Create the model instance
                start_time = time.time()
                net2 = DCSM(lstm_params, gcn_params, linear_params, classify_params, dropout=0.5)

                        # Train and evaluate the model
                test_loss, test_auc, test_aupr, test_acc, test_recall, test_precision, test_f1 = classify_train(
                            net2, Epoch_num, Batch_size, learning_rate, train_loader, test_loader, loss_weights_sets)

                        # Record the runtime for this fold
                end_time = time.time()
                fold_time = end_time - start_time
                fold_times.append(fold_time)

                        # Store the fold results
                results.append({
                            "Fold": fold_number,
                            "Classification_Label": classification_label,
                            "Taxonomy": taxonomy,
                            "AUC": test_auc,
                            "AUPR": test_aupr,
                            "ACC": test_acc,
                            "Recall": test_recall,
                            "Precision": test_precision,
                            "F1-Score": test_f1,
                            "Runtime (s)": fold_time,                            
                        })

                    # Compute and append the average results, including weights
            mean_results = {
                        "Fold": "Average",
                        "Classification_Label": classification_label,
                        "Taxonomy": taxonomy,
                        "AUC": np.mean([res["AUC"] for res in results]),
                        "AUPR": np.mean([res["AUPR"] for res in results]),
                        "ACC": np.mean([res["ACC"] for res in results]),
                        "Recall": np.mean([res["Recall"] for res in results]),
                        "Precision": np.mean([res["Precision"] for res in results]),
                        "F1-Score": np.mean([res["F1-Score"] for res in results]),
                        "Runtime (s)": np.mean(fold_times),
                    }
                    
                                    # Append the mean results to the results list
            results.append(mean_results)
                    
            for res in results:
                all_results.append({
                            **res, 
                            "Dataset": dataset_name  #
                        })
                    # Append all fold results to the final results
            dataset_results.extend(results)

                    # Save the intermediate results
            with open(temp_save_path, 'w') as f:
                json.dump(dataset_results, f)
            with open(global_temp_save_path, 'w') as f:
                json.dump(all_results, f)
                        # Save the results to Excel
    save_path = f"/{dataset_name}/Classify_result_summary.xlsx"
    save_results_to_excel(dataset_results, save_path)  
save_path_all = f"/Classify_results_summary.xlsx"
save_results_to_excel(all_results, save_path_all)

Running classification for BONUS-CF,Cysticfibrosis, taxonomy P and {'w_rec': 0.1, 'w_dag': 0.0001, 'w_dynamic': 0.01, 'w_consistency': 1e-05}...
Dataset BONUS-CF, Running Fold 1 for Cysticfibrosis and taxonomy P...
Epoch: 0, train_Loss: 0.23039, train_AUC: 0.38929, train_AUPR: 0.85614, train_ACC: 0.88506, train_Precision: 0.99351,train_ Recall: 0.88953, train_F1: 0.93865
  val_Loss: 0.22439, val_AUC: 0.41026, val_AUPR: 0.40909, val_ACC: 0.88213, val_Precision: 0.35897, val_Recall: 0.93333, val_F1: 0.51852
Epoch: 50, train_Loss: 0.19113, train_AUC: 0.63961, train_AUPR: 0.92567, train_ACC: 0.56322, train_Precision: 0.53896,train_ Recall: 0.94318, train_F1: 0.68595
  val_Loss: 0.19035, val_AUC: 1.00000, val_AUPR: 1.00000, val_ACC: 1.00000, val_Precision: 1.00000, val_Recall: 1.00000, val_F1: 1.00000
Epoch: 100, train_Loss: 0.15920, train_AUC: 0.99351, train_AUPR: 0.99917, train_ACC: 0.95977, train_Precision: 0.95455,train_ Recall: 1.00000, train_F1: 0.97674
  val_Loss: 0.14781, val_AUC: 1

In [21]:
all_results

[{'Fold': 1,
  'Classification_Label': 'Cysticfibrosis',
  'Taxonomy': 'P',
  'AUC': 1.0,
  'AUPR': 1.0,
  'ACC': 1.0,
  'Recall': 1.0,
  'Precision': 1.0,
  'F1-Score': 1.0,
  'Runtime (s)': 37.77387022972107,
  'Dataset': 'BONUS-CF'},
 {'Fold': 2,
  'Classification_Label': 'Cysticfibrosis',
  'Taxonomy': 'P',
  'AUC': 1.0,
  'AUPR': 0.9999999999999998,
  'ACC': 1.0,
  'Recall': 1.0,
  'Precision': 1.0,
  'F1-Score': 1.0,
  'Runtime (s)': 37.470370054244995,
  'Dataset': 'BONUS-CF'},
 {'Fold': 3,
  'Classification_Label': 'Cysticfibrosis',
  'Taxonomy': 'P',
  'AUC': 1.0,
  'AUPR': 1.0,
  'ACC': 1.0,
  'Recall': 1.0,
  'Precision': 1.0,
  'F1-Score': 1.0,
  'Runtime (s)': 37.187827348709106,
  'Dataset': 'BONUS-CF'},
 {'Fold': 4,
  'Classification_Label': 'Cysticfibrosis',
  'Taxonomy': 'P',
  'AUC': 1.0,
  'AUPR': 0.9999999999999998,
  'ACC': 1.0,
  'Recall': 1.0,
  'Precision': 1.0,
  'F1-Score': 1.0,
  'Runtime (s)': 37.23780703544617,
  'Dataset': 'BONUS-CF'},
 {'Fold': 5,
  'Clas